In [1]:
!git clone https://github.com/bhujbalatharva252-sketch/Recommendation-Systems-benchmarking

Cloning into 'Recommendation-Systems-benchmarking'...
remote: Enumerating objects: 102, done.
remote: Counting objects: 100% (46/46), done.
remote: Compressing objects: 100% (41/41), done.
remote: Total 102 (delta 18), reused 22 (delta 4), pack-reused 56 (from 1)
Receiving objects: 100% (102/102), 43.06 MiB | 25.54 MiB/s, done.
Resolving deltas: 100% (33/33), done.


In [2]:
%cd /content/Recommendation-Systems-benchmarking/

/content/Recommendation-Systems-benchmarking


# Load data and candidate set

In [3]:
import pandas as pd
import numpy as np
import pickle

# Load train and test data
train= pd.read_csv("data/processed/train.csv")
test= pd.read_csv("data/processed/test.csv")

# Load movie metadata
movies=pd.read_csv("data/raw/movies.dat",
    sep="::",
    engine="python",
    names=["MovieID", "Title", "Genres"],
    encoding="latin-1")

# Load fixed candidate set
candidate_sets=pd.read_pickle("data/processed/candidate_sets_100.pkl")

print("Data loaded successfully")

Data loaded successfully


In [4]:
from llm_utils_py import (create_prompt,load_llm,generate_recommendations)

# Gemma-7B Instruct

In [5]:
from huggingface_hub import login

login()

In [6]:

gemma_name = "google/gemma-7b-it"

gemma_tokenizer, gemma_model = load_llm(gemma_name)

print("Gemma loaded")

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/34.2k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.5MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

Gemma loaded


In [7]:
type(candidate_sets)

dict

In [8]:
candidate_sets[1].keys()

dict_keys(['test_item', 'items'])

In [9]:
user_id = 1

candidate_ids = candidate_sets[user_id]["items"]

candidate_movies = movies[
    movies["MovieID"].isin(candidate_ids)
]

In [10]:
prompt = create_prompt(
    user_id=user_id,
    train=train,
    movies=movies,
    candidate_movies=candidate_movies
)

print(prompt)



You are a movie recommendation system.
Your task is to rank movies based on the user's previous preferences.

User's watched movies:
Back to the Future (1985) (Comedy|Sci-Fi) Rating: 5
Cinderella (1950) (Animation|Children's|Musical) Rating: 5
Christmas Story, A (1983) (Comedy|Drama) Rating: 5
Last Days of Disco, The (1998) (Drama) Rating: 5
Saving Private Ryan (1998) (Action|Drama|War) Rating: 5
Awakenings (1990) (Drama) Rating: 5
One Flew Over the Cuckoo's Nest (1975) (Drama) Rating: 5
Rain Man (1988) (Drama) Rating: 5
Sound of Music, The (1965) (Musical) Rating: 5
Ben-Hur (1959) (Action|Adventure|Drama) Rating: 5
Apollo 13 (1995) (Drama) Rating: 5
Mary Poppins (1964) (Children's|Comedy|Musical) Rating: 5
Dumbo (1941) (Animation|Children's|Musical) Rating: 5
Schindler's List (1993) (Drama|War) Rating: 5
Beauty and the Beast (1991) (Animation|Children's|Musical) Rating: 5
Bug's Life, A (1998) (Animation|Children's|Comedy) Rating: 5
Toy Story (1995) (Animation|Children's|Comedy) Rati

# Evaluation Loop

In [11]:
import os
import time
import pickle
from google.colab import drive

# Mount Google Drive
drive.mount("/content/drive")

# Path to save checkpoint
checkpoint_path = "/content/drive/MyDrive/gemma_checkpoint.pkl"

# Load checkpoint if it exists

if os.path.exists(checkpoint_path):

    with open(checkpoint_path, "rb") as f:
        gemma_results = pickle.load(f)
    print(f"Loaded checkpoint with {len(gemma_results)} users.")

else:
    gemma_results = {}
    print("No checkpoint found. Starting from scratch.")

# Evaluation loop

start_time = time.time()

total_users = len(candidate_sets)

for count, (user_id, data) in enumerate(candidate_sets.items(), start=1):

    # Skip already processed users
    if user_id in gemma_results:
        continue

    print(f"Processing user {count}/{total_users}")

    # Convert candidate MovieIDs to dataframe
    candidate_movies = movies[
        movies["MovieID"].isin(data["items"])
    ]

    # Create prompt
    prompt = create_prompt(
        user_id=user_id,
        train=train,
        movies=movies,
        candidate_movies=candidate_movies
    )

    # Skip users with no history
    if prompt is None:
        continue

    # Generate recommendation using Gemma
    response = generate_recommendations(prompt,gemma_tokenizer,gemma_model)

    # Store result
    gemma_results[user_id] = response

    # Save checkpoint every 50 users
    if len(gemma_results)%50==0:

        with open(checkpoint_path, "wb") as f:
            pickle.dump(gemma_results,f)

        elapsed = (time.time() - start_time) / 60

        print(f"Checkpoint saved ({len(gemma_results)} users)")

        print(f"Elapsed time: {elapsed:.2f} minutes")

# Save final results
with open(checkpoint_path, "wb") as f:
    pickle.dump(gemma_results,f)

print("----------------------------------")
print("Gemma evaluation completed")
print(f"Total users processed: {len(gemma_results)}")
print("Results saved to:")
print(checkpoint_path)

Mounted at /content/drive
Loaded checkpoint with 5650 users.
Processing user 5651/6040
Processing user 5652/6040
Processing user 5653/6040
Processing user 5654/6040
Processing user 5655/6040
Processing user 5656/6040
Processing user 5657/6040
Processing user 5658/6040
Processing user 5659/6040
Processing user 5660/6040
Processing user 5661/6040
Processing user 5662/6040
Processing user 5663/6040
Processing user 5664/6040
Processing user 5665/6040
Processing user 5666/6040
Processing user 5667/6040
Processing user 5668/6040
Processing user 5669/6040
Processing user 5670/6040
Processing user 5671/6040
Processing user 5672/6040
Processing user 5673/6040
Processing user 5674/6040
Processing user 5675/6040
Processing user 5676/6040
Processing user 5677/6040
Processing user 5678/6040
Processing user 5679/6040
Processing user 5680/6040
Processing user 5681/6040
Processing user 5682/6040
Processing user 5683/6040
Processing user 5684/6040
Processing user 5685/6040
Processing user 5686/6040
Pro

# Parse LLM Output

In [12]:
import re

def parse_llm_ranking(response_text):
    """
    Extract MovieIDs from LLM response.

    Example output:
    1. MovieID: 123
    2. MovieID: 456

    Returns:
    [123,456]
    """

    movie_ids=re.findall(r"MovieID:\s*(\d+)",response_text)

    # Convert string IDs to integers
    movie_ids=[int(mid) for mid in movie_ids]

    return movie_ids

# Scoring function

In [13]:
def llm_score_fn(user_id, items):

    # Get LLM response for this user
    response=gemma_results[user_id]

    # Parse ranking
    ranked_movies=parse_llm_ranking(response)

    # Assign scores
    movie_scores={}

    total_movies=len(ranked_movies)

    for position,movie_id in enumerate(ranked_movies):

        movie_scores[movie_id]=total_movies-position


    # Return scores in same order as candidate items
    scores=[]

    for movie_id in items:
        scores.append(movie_scores.get(movie_id, 0))

    return scores

# Metrics

In [14]:
# Hit@10=Checks whether the true test item appears in the top 10 recommended items.

def hit_at_k(ranked_items, test_item, k=10):
    if test_item in ranked_items[:k]:
        return 1.0

    return 0.0

# NDCG@10 = Measures ranking quality. Higher score is given when the true item appears higher in the top 10.

def ndcg_at_k(ranked_items, test_item, k=10):
    top_k=ranked_items[:k]

    # Check whether the true item is recommended
    if test_item in top_k:

        # Get position of true item (starting from 1)
        rank=top_k.index(test_item)+1

        # Discount score based on ranking position
        return 1.0/np.log2(rank+1)

    return 0.0

# Evaluate

In [15]:
# Evaluate a recommendation model.
    # This function will be used for all recommendation models.
    # It calculates the average Hit@10 and NDCG@10 using the fixed candidate set
def evaluate_model(score_fn, candidate_sets, k=10):
    hits=[]
    ndcgs=[]

    # Evaluate every user
    for user_id,candidate in candidate_sets.items():

        # Get 100 candidate movies
        items= candidate["items"]

        # Actual movie from test set
        test_item= candidate["test_item"]


        # Get model prediction scores
        scores= score_fn(user_id, items)

         # Rank movies according to predicted scores
        ranked_items= [item for item, score in sorted(
                zip(items, scores),
                key=lambda x: x[1],
                reverse=True)]


        # Calculate metrics
        hits.append(hit_at_k(ranked_items, test_item, k))

        ndcgs.append(
            ndcg_at_k(ranked_items, test_item, k)
        )
    # Return average performance over all users
    return {
        "Hit@10": np.mean(hits),
        "NDCG@10": np.mean(ndcgs)
    }

In [17]:
gemma_results_metrics=evaluate_model(llm_score_fn,candidate_sets,k=10)

print("Gemma Results")
print(f"Hit@10 : {gemma_results_metrics['Hit@10']:.4f}")
print(f"NDCG@10: {gemma_results_metrics['NDCG@10']:.4f}")

Gemma Results
Hit@10 : 0.0919
NDCG@10: 0.0443
